In [ ]:
import pyvo
import numpy as np
from astropy.table import Table, MaskedColumn
from crossmatching.catalogs.base import CatalogBase


class MyTAPCatalog(CatalogBase):
    # --- Column name mapping ---
    # Tell the crossmatcher which column in your table holds each concept.
    ra_key = "ra"
    dec_key = "dec"
    hostname_key = "host"       # star name column in your catalog
    planet_uid = "planet_name" # unique identifier for each planet row
    pm_key = None               # set to a column name if your catalog has proper motion
    pmerr_key = None            # set to the PM error column if available

    TAP_URL = "http://my-tap-service.example.org/tap"
    QUERY   = "SELECT * FROM my_schema.my_planets"

    def download(self) -> Table:
        """Query the remote TAP service and return the raw table."""
        service = pyvo.dal.TAPService(self.TAP_URL)
        return service.search(self.QUERY).to_table()

    def preprocess(self, table: Table) -> Table:
        """Add derived columns. Called automatically by load() after download() or load_raw().

        Option A: your catalog has a direct epoch column — rename it here.
        # table["coord_epoch"] = table["ref_epoch"]

        Option B: compute epoch from metadata (e.g. all Gaia → 2016.0).
        # table["coord_epoch"] = MaskedColumn([2016.0] * len(table), name="coord_epoch")

        If you don't add "coord_epoch", the crossmatcher falls back to the
        unknown_default search radius (50 arcsec) for every row.
        """
        return table


# --- Cache the raw data to disk (avoids the slow TAP query on every run) ---
# MyTAPCatalog().save_raw("my_cache.txt")   # run once to download

# --- Plug it in ---
from crossmatching import Crossmatcher
from crossmatching.id_suppliers.simbad import SimbadIdSupplier

cm = Crossmatcher(
    catalog=MyTAPCatalog(),
    id_supplier=SimbadIdSupplier(),
)

# Load from cache (raw file → preprocess), or omit from_file to query live
# cm.load_catalog(from_file="my_cache.txt")

# Then use exactly as normal:
# cm.load_alternate_ids(input["star_name"].tolist())
# result = cm.combined_crossmatch(input)
